In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

directory = "../reports/arena/"

# 48854

In [ ]:
md = "# Report Arena\n\n"

for idx, _fname in enumerate([p.name for p in Path(directory).iterdir()]):
    if idx > 0:
        break
    path = str(Path(directory) / "bc_74_v1_hs512_RSP_50_RND_1_TESTING5.csv")  # _fname
    md += f"## File: {path}\n\n"
    if path.endswith(".md"):
        continue
    evaluation_data = pd.read_csv(path)
    print(path)

    infos = path.removesuffix(".csv").split("/")[-1].split("_")

    if len(infos) == 3:
        data_player, data_enemy, _ = infos

        player = "DummyPlayer" if data_player == "DP" else "RandomSauPlayer"

        player_2 = "DummyPlayer" if data_enemy == "DP" else "RandomSauPlayer"

        md += f"## {data_player} vs. {data_enemy}\n\n"

    if len(infos) == 4:
        data_player = infos[0] + "_" + infos[1]
        data_enemy = infos[2]
        player = "RLPlayer"

        player_2 = "DummyPlayer" if data_enemy == "DP" else "RandomSauPlayer"

        md += f"## {data_player} vs. {data_enemy}\n\n"

    if len(infos) == 5:
        data_player, input_size, ver, data_enemy, _ = infos
        if data_player.lower().startswith("bc"):
            player = "BCPlayer"
        if data_player.lower().startswith(("rl", "reward")):
            player = "RLPlayer"

        data_player = data_player + "_" + input_size + "_" + ver
        player_2 = "DummyPlayer" if data_enemy == "DP" else "RandomSauPlayer"

        md += f"## {data_player} vs. {data_enemy}\n\n"

    if len(infos) == 6:
        data_player, input_size, ver, hidden_size, data_enemy, _ = infos
        player = "BCPlayer"

        data_player = data_player + "_" + input_size + "_" + ver + "_" + hidden_size
        player_2 = "DummyPlayer" if data_enemy == "DP" else "RandomSauPlayer"

        md += f"## {data_player} vs. {data_enemy}\n\n"

    player = "BCPlayer"
    player_2 = "RandomSauPlayer"

    # Extract numeric IDs
    evaluation_data["PLAYER_ID"] = evaluation_data["UID"].astype(str).str.extract(r"Player ID (\d+)").astype(int)
    evaluation_data["TEAMMEMBER_ID"] = (
        evaluation_data["TEAMMEMBER"].astype(str).str.extract(r"Player ID (\d+)").astype(int)
    )

    # Map to net-size labels
    mapping = {0: 46, 1: 46, 2: 70, 3: 70}
    evaluation_data["PLAYER_NET"] = evaluation_data["PLAYER_ID"].map(mapping).astype(str).radd("BC_")
    evaluation_data["TEAMMEMBER_NET"] = evaluation_data["TEAMMEMBER_ID"].map(mapping).astype(str).radd("BC_")

    # (Optional) drop old splits
    # evaluation_data.drop(columns=['PLAYER_TYPE','TEAMMEMBER_TYPE'], inplace=True)

    # display(evaluation_data[['UID','PLAYER_ID','PLAYER_NET','TEAMMEMBER','TEAMMEMBER_ID','TEAMMEMBER_NET']].head())

    evaluation_data["WIN/LOSE"] = np.where(evaluation_data["WIN/LOSE"] > 0, 1, 0)

    split_cols = evaluation_data["PLAYER"].str.split(",", n=1, expand=True)[1]
    split_cols.columns = "PLAYER TYPE"

    # display(split_cols)

    pos = evaluation_data.columns.get_loc("PLAYER") + 1
    evaluation_data.insert(pos, "PLAYER TYPE (GAME)", split_cols)

    # display(evaluation_data.head())

    # display(f"Total Gamesplayed: {evaluation_data.shape[0]}")

    md += f"Total Gamesplayed: {evaluation_data.shape[0]}\n\n"

    ######### Player 1

    df_player = evaluation_data[evaluation_data["PLAYER_TYPE"] == player]
    # df_player = evaluation_data[evaluation_data['PLAYER_NET'] == player]

    games = df_player.shape[0]
    # display("Overall Gamesplayed by " + player + f": {games}")
    winrate = np.round(df_player["WIN/LOSE"].sum() / games, 6)
    # display(f"Winrate: {winrate}")
    # display(f"Teamdist: {df_player['TEAM'].sum() / games}")

    md += "### " + "Overall Gamesplayed by " + player + f": {games}" + "\n\n"
    md += f"Winrate: {winrate}" + "\n\n  "
    md += f"Teamdist: {df_player['TEAM'].sum() / games}" + "\n\n  "

    #########

    df_same_player = df_player[df_player["TEAMMEMBER_TYPE"] == player]
    # df_same_player = df_player[df_player['TEAMMEMBER_NET'] == player]
    # display(f"Same Player Team Games: {df_same_player.shape[0]}")
    winrate_same_team = np.round(df_same_player["WIN/LOSE"].sum() / df_same_player.shape[0], 6)
    # display(f"Same Player Team Winrate: {winrate_same_team}")

    md += f"Same Player Team Games: {df_same_player.shape[0]}" + "\n\n  "
    md += f"Same Player Team Winrate: {winrate_same_team}" + "\n\n  "

    #########
    # df_spieler_same_team = df_player[(df_player['TEAM'] == 0) & (df_player['TEAMMEMBER_NET'] == player)]
    df_spieler_same_team = df_player[(df_player["TEAM"] == 0) & (df_player["TEAMMEMBER_TYPE"] == player)]
    # display(f"Same Player Team and 'Ansage' Games: {df_spieler_same_team.shape[0]}")

    winrate_same_team_ansage = np.round(df_spieler_same_team["WIN/LOSE"].sum() / df_spieler_same_team.shape[0], 6)
    # display(f"Same Player Team and 'Ansage' Winrate: {winrate_same_team_ansage}")

    # display(f"Total Gamesplayed: {evaluation_data.shape[0]}")

    md += f"Same Player Team and 'Ansage' Games: {df_spieler_same_team.shape[0]}" + "\n\n  "
    md += f"Same Player Team and 'Ansage' Winrate: {winrate_same_team_ansage}" + "\n\n  "

    ######### Player 2

    df_player = evaluation_data[evaluation_data["PLAYER_TYPE"] == player_2]
    # df_player = evaluation_data[evaluation_data['PLAYER_NET'] == enemy]
    games = df_player.shape[0]
    # display("Overall Gamesplayed by " + player_2 + f": {games}")
    winrate = np.round(df_player["WIN/LOSE"].sum() / games, 6)
    # display(f"Winrate: {winrate}")
    # display(f"Teamdist: {df_player['TEAM'].sum() / games}")

    md += "### " + "Overall Gamesplayed by " + player_2 + f": {games}" + "\n\n  "
    md += f"Winrate: {winrate}" + "\n\n"
    md += f"Teamdist: {df_player['TEAM'].sum() / games}" + "\n\n  "

    #########

    df_same_player = df_player[df_player["TEAMMEMBER_TYPE"] == player_2]
    # df_same_player = df_player[df_player['TEAMMEMBER_NET'] == player_2]
    # display(f"Same Player Team Games: {df_same_player.shape[0]}")
    winrate_same_team = np.round(df_same_player["WIN/LOSE"].sum() / df_same_player.shape[0], 6)
    # display(f"Same Player Team Winrate: {winrate_same_team}")

    md += f"Same Player Team Games: {df_same_player.shape[0]}" + "\n\n  "
    md += f"Same Player Team Winrate: {winrate_same_team}" + "\n\n  "

    #########

    # df_spieler_same_team = df_player[(df_player['TEAM'] == 0) & (df_player['TEAMMEMBER_NET'] == player_2)]
    df_spieler_same_team = df_player[(df_player["TEAM"] == 0) & (df_player["TEAMMEMBER_TYPE"] == player_2)]
    # display(f"Same Player Team and 'Ansage' Games: {df_spieler_same_team.shape[0]}")

    winrate_same_team_ansage = np.round(df_spieler_same_team["WIN/LOSE"].sum() / df_spieler_same_team.shape[0], 6)
    # display(f"Same Player Team and 'Ansage' Winrate: {winrate_same_team_ansage}")

    md += f"Same Player Team and 'Ansage' Games: {df_spieler_same_team.shape[0]}" + "\n\n  "
    md += f"Same Player Team and 'Ansage' Winrate: {winrate_same_team_ansage}" + "\n\n  "


with Path("../reports/arena/arena_easy.md").open("w", encoding="utf-8") as f:
    f.write(md)